# nets

> Neural networks


In [1]:
#| default_exp nets

In [2]:
#| hide
from nbdev.showdoc import *

In [3]:
#| hide
# from IPython.display import clear_output, DisplayHandle


# def update_patch(self, obj):
#     clear_output(wait=True)
#     self.display(obj)


# DisplayHandle.update = update_patch

In [4]:
from torch import randn as torch_randn
from fastai.vision.all import test_eq

In [5]:
#| export

from fastai.vision.all import ConvLayer, Lambda, MaxPool, NormType, nn, np
from torch import cat as torch_cat
import torch.nn.functional as F
from bioMONAI.core import attributesFromDict

## DnCNN


In [6]:
#| export

class DnCNN(nn.Module):
    def __init__(self, channels, num_of_layers=9, features=64, kernel_size=3):
        super(DnCNN, self).__init__()
        padding = 1
        layers = []
        layers.append(ConvLayer(channels, features,
                      ks=kernel_size, padding=padding, norm_type=None))
        for _ in range(num_of_layers-2):
            layers.append(ConvLayer(features, features,
                          ks=kernel_size, padding=padding))
        layers.append(nn.Conv2d(in_channels=features, out_channels=channels,
                      kernel_size=kernel_size, padding=padding, bias=False))
        self.dncnn = nn.Sequential(*layers)

    def forward(self, x):
        residual = self.dncnn(x)
        denoised = x - residual
        return denoised

In [7]:
x = torch_randn(16, 1, 32, 64)

tst = DnCNN(1)
test_eq(tst(x).shape, x.shape)

## UNet


### Convolutional sub-unit


In [8]:
#| export
def SubNetConv(ks=3,
               stride=1,
               padding=None,
               bias=None,
               ndim=2,
               norm_type=NormType.Batch,
               bn_1st=True,
               act_cls=nn.ReLU,
               transpose=False,
               init='auto',
               xtra=None,
               bias_std=0.01,
               dropout=0.0,
               ):

    def _conv(n_in, n_out, n_conv=1):
        s = ConvLayer(n_in, n_out, ks=ks, stride=stride, padding=padding, bias=bias, ndim=ndim, norm_type=norm_type, bn_1st=bn_1st,
                      act_cls=act_cls, transpose=transpose, init=init, xtra=xtra, bias_std=bias_std)
        if dropout is not None and dropout > 0:
            s = nn.Sequential(s, nn.Dropout(dropout))
        for _ in range(n_conv-1):
            t = ConvLayer(n_out, n_out, ks=ks, stride=stride, padding=padding, bias=bias, ndim=ndim, norm_type=norm_type, bn_1st=bn_1st,
                          act_cls=act_cls, transpose=transpose, init=init, xtra=xtra, bias_std=bias_std)
            if dropout is not None and dropout > 0:
                t = nn.Sequential(t, nn.Dropout(dropout))
            s = nn.Sequential(s, t)
        return s

    return _conv

In [9]:
x = torch_randn(16, 1, 32, 64, 64)
xdim = len(x.shape)-2

# reduce
tst = SubNetConv(3, padding=1, stride=2, ndim=xdim,
                 norm_type=NormType.Batch, dropout=.1)(1, 2, 2)
y = tst(x)
test_eq(y.shape, [16, 2, 8, 16, 16])
print(tst)
# upsample
tst = SubNetConv(ks=4, padding=0, stride=4, ndim=xdim, norm_type=NormType.Batch,
                 transpose=True)(2, 1)  # to double the size, the kernel cannot be odd
test_eq(tst(y).shape, x.shape)
del y

Sequential(
  (0): Sequential(
    (0): ConvLayer(
      (0): Conv3d(1, 2, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
      (1): BatchNorm3d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
    )
    (1): Dropout(p=0.1, inplace=False)
  )
  (1): Sequential(
    (0): ConvLayer(
      (0): Conv3d(2, 2, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
      (1): BatchNorm3d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
    )
    (1): Dropout(p=0.1, inplace=False)
  )
)


### UNet block


In [10]:
#| export
class _Net_recurse(nn.Module):
    def __init__(self,
                 depth=4,						# depth of the UNet network
                 mult_chan=32,					# number of filters at first layer
                 in_channels=1,					# number of input channels
                 kernel_size=3,					# kernel size of convolutional layers
                 ndim=2,							# number of spatial dimensions of the input data
                 n_conv_per_depth=2,				# number of convolutions per layer
                 activation=nn.ReLU,				# activation function used in convolutional layers
                 norm_type=NormType.Batch,
                 dropout=0.0,
                 pool=MaxPool,
                 pool_size=2,
                 ):
        """Class for recursive definition of U-network.p

        Parameters:
        in_channels - (int) number of channels for input.
        mult_chan - (int) factor to determine number of output channels
        depth - (int) if 0, this subnet will only be convolutions that double the channel count.
        """
        super().__init__()
        # Parameters
        self.depth = depth
        n_out = in_channels*mult_chan

        # Layer types
        Pooling = pool(ks=pool_size, ndim=ndim)
        UpSample = nn.Upsample(scale_factor=pool_size, mode='nearest')
        SubNet_Conv = SubNetConv(ks=kernel_size, stride=1, padding=None, bias=None, ndim=ndim, norm_type=norm_type,
                                 bn_1st=True, act_cls=activation, transpose=False, dropout=dropout)

        # Blocks
        self.sub_conv_more = SubNet_Conv(in_channels, n_out, n_conv_per_depth)
        if self.depth > 0:
            in_channels = n_out
            mult_chan = 2
            depth = (self.depth - 1)
            self.sub_u = nn.Sequential(Pooling,                                                         # layer reducing the image size (usually a pooling layer)
                                       _Net_recurse(depth, mult_chan, in_channels, kernel_size,
                                                    ndim, n_conv_per_depth, activation, norm_type,
                                                    dropout, pool, pool_size),                          # lower unet level
                                       # layer increasing the image size (usually an upsampling layer)
                                       UpSample,
                                       )
            self.sub_conv_less = SubNet_Conv(3*n_out, n_out, n_conv_per_depth)

    def forward(self, x):
        if self.depth == 0:
            return self.sub_conv_more(x)
        else:  # depth > 0
            # convolutions with increasing number of channels
            x_conv_more = self.sub_conv_more(x)
            x_from_sub_u = self.sub_u(x_conv_more)
            # concatenate the upsampled outputs of the lower level with the outputs of the next level in size
            x_cat = torch_cat((x_from_sub_u, x_conv_more), 1)
            # convolutions with decreasing number of channels
            x_conv_less = self.sub_conv_less(x_cat)
        return x_conv_less

### Recursive UNet


In [11]:
#| export
class UNet(nn.Module):
    def __init__(self,
                 depth=4,						# depth of the UNet network
                 mult_chan=32,					# number of filters at first layer
                 in_channels=1,					# number of input channels
                 out_channels=1,					# number of output channels
                 last_activation=None,			# last activation before final result
                 kernel_size=3,					# kernel size of convolutional layers
                 ndim=2,							# number of spatial dimensions of the input data
                 n_conv_per_depth=2,				# number of convolutions per layer
                 activation='ReLU',				# activation function used in convolutional layers
                 norm_type=NormType.Batch,
                 dropout=0.0,
                 pool=MaxPool,
                 pool_size=2,
                 residual=False,
                 prob_out=False,
                 eps_scale=1e-3,
                 ):
        super().__init__()
        last_activation = getattr(F, f"{activation.lower()}") if last_activation == None else getattr(
            F, f"{last_activation.lower()}")
        activation = getattr(nn, f"{activation}")
        attributesFromDict(locals())		# stores all the input parameters in self

        self.net_recurse = _Net_recurse(depth, mult_chan, in_channels, kernel_size, ndim,
                                        n_conv_per_depth, activation, norm_type, dropout, pool, pool_size)
        self.conv_out = ConvLayer(mult_chan*in_channels, out_channels, ndim=ndim,
                                  ks=kernel_size, norm_type=None, act_cls=None, padding=1)

    def forward(self, x):
        x_rec = self.net_recurse(x)
        final = self.conv_out(x_rec)

        if self.residual:
            if not (self.out_channels == self.in_channels):
                raise ValueError(
                    "number of input and output channels must be the same for a residual net.")
            final = final + x
        final = self.last_activation(final)

        if self.prob_out:
            scale = ConvLayer(self.out_channels, self.out_channels,
                              ndim=self.ndim, ks=1, norm_type=None, act_cls=nn.Softplus)(x_rec)
            scale = Lambda(lambda x: x+np.float32(self.eps_scale))(scale)
            final = torch_cat((final, scale), 1)

        return final

In [13]:
x = torch_randn(16, 1, 32, 64, 64)
xdim = len(x.shape)-2

tst = UNet(depth=1, ndim=xdim, n_conv_per_depth=1, residual=True)
mods = list(tst.children())
print(mods)
test_eq(tst(x).shape, [16, 1, 32, 64, 64])

[_Net_recurse(
  (sub_conv_more): ConvLayer(
    (0): Conv3d(1, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
    (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (sub_u): Sequential(
    (0): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (1): _Net_recurse(
      (sub_conv_more): ConvLayer(
        (0): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
      )
    )
    (2): Upsample(scale_factor=2.0, mode='nearest')
  )
  (sub_conv_less): ConvLayer(
    (0): Conv3d(96, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
    (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
), ConvLayer(
  (0): Conv3d(32, 1, kernel_size=(3, 3, 3), stride=(1, 

## DeepLab v3

In [ ]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

class DeepLabV3Plus(nn.Module):
    def __init__(self, num_classes=21, input_size=(256, 256)):
        super(DeepLabV3Plus, self).__init__()
        # Store the user-defined input size for later use
        self.input_size = input_size
        
        # Load the pre-trained ResNet50 model
        self.backbone = models.resnet50(pretrained=True)
        
        # Replace the last layer of the backbone to match the number of classes
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        
        # ASPP (Atrous Spatial Pyramid Pooling)
        self.aspp = nn.Sequential(
            nn.Conv2d(num_features, 256, kernel_size=1, stride=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            ASPPPooling(num_features, 256),
            nn.Conv2d(256, num_classes, kernel_size=1, stride=1)
        )
        
        # Decoder (upsampling + convolution)
        self.decoder = nn.Sequential(
            nn.Conv2d(num_features + 256, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(256),
            nn.Conv2d(256, num_classes, kernel_size=3, padding=1)
        )
        
    def forward(self, x):
        # Ensure the input size matches the defined input size
        if x.shape[2:] != self.input_size:
            raise ValueError("Input size must be {}x{}".format(*self.input_size))
        
        # Extract features from the backbone
        feat = self.backbone.features(x)
        
        # Apply ASPP to get high-level feature maps
        aspp_out = self.aspp(feat[-1])
        
        # Upsample and concatenate with low-level features
        x = F.interpolate(x, size=feat[-1].size()[2:], mode='bilinear', align_corners=False)
        concat_feat = torch.cat([x, aspp_out], dim=1)

        # Apply decoder to get the final segmentation output
        out = self.decoder(concat_feat)
        
        return out

class ASPPPooling(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ASPPPooling, self).__init__()
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1)
        )
        
    def forward(self, x):
        pooled = self.gap(x)
        return F.interpolate(pooled, size=x.size()[2:], mode='bilinear', align_corners=False)


In [ ]:
# Example usage
# Create the model and move it to GPU if available
model = DeepLabV3Plus(num_classes=21, input_size=(512, 512)).cuda()

# Define a dummy input tensor (e.g., batch size of 1, RGB images with defined resolution)
x = torch.randn(1, 3, 512, 512).cuda()

# Forward pass through the model
output = model(x)
print(output.shape)  # Should be (1, num_classes, defined resolution)



---

In [ ]:
from typing import Sequence, Union

import torch
import torch.nn as nn
import torch.nn.functional as F
from monai.networks.blocks import Convolution, UpSample
from monai.networks.layers.factories import Act, Conv, Dropout, Norm, Pool
from monai.utils import ensure_tuple_rep
import math
from typing import Optional, Sequence, Type, Union


In [ ]:

# Function to apply fixed padding to inputs for convolutional layers
def fixed_padding(inputs, kernel_size, dilation):
    # Calculate the effective kernel size considering dilation
    kernel_size_effective = kernel_size + (kernel_size - 1) * (dilation - 1)
    
    # Calculate the total padding required on each side of the input
    pad_total = kernel_size_effective - 1
    
    # Split the total padding equally between the beginning and end
    pad_beg = pad_total // 2
    pad_end = pad_total - pad_beg
    
    # Apply padding to the inputs using PyTorch's F.pad function
    padded_inputs = F.pad(inputs, (pad_beg, pad_end, pad_beg, pad_end, pad_beg, pad_end))
    
    return padded_inputs


In [ ]:
class SeparableConv2d_same(nn.Module):
    def __init__(self, inplanes, planes, kernel_size=3, stride=1, dilation=1, bias=False,norm=None):
        super(SeparableConv2d_same, self).__init__()
        dim = 3
        self.kernel_size = kernel_size
        self.dilation = dilation
        self.conv1 = Convolution(dim, inplanes, inplanes, kernel_size=kernel_size,
                                 groups=inplanes, padding=0, dilation=dilation, bias=bias, strides=stride)
        if norm == None:
           self.pointwise = Convolution(dim, inplanes, planes, kernel_size=1, strides=1,
                                     padding=0, dilation=1, groups=1, bias=bias)
        else:
           self.pointwise = Convolution(dim, inplanes, planes, kernel_size=1, strides=1,
                                     padding=0, dilation=1, groups=1, bias=bias,norm=Norm.BATCH)

    def forward(self, x):
        x = fixed_padding(x, self.kernel_size, self.dilation)
        x = self.conv1(x)
        x = self.pointwise(x)
        return x


In [ ]:
class Block(nn.Module):
    def __init__(self, inplanes, planes, reps, stride=1, dilation=1, start_with_relu=True, grow_first=True, is_last=False):
        super(Block, self).__init__()
        dim = 3
        if planes != inplanes or stride != 1:
            self.skip = Convolution(dim, inplanes, planes, kernel_size=1, bias=False, strides=stride,norm=Norm.BATCH)
        else:
            self.skip = None

        self.relu = Act[Act.RELU](inplace=True)
        rep = []

        filters = inplanes
        if grow_first:
            rep.append(self.relu)
            rep.append(SeparableConv2d_same(inplanes, planes, 3, stride=1, dilation=dilation,norm=Norm.BATCH))
            filters = planes

        for i in range(reps - 1):
            rep.append(self.relu)
            rep.append(SeparableConv2d_same(filters, filters, 3, stride=1, dilation=dilation,norm=Norm.BATCH))

        if not grow_first:
            rep.append(self.relu)
            rep.append(SeparableConv2d_same(inplanes, planes, 3, stride=1, dilation=dilation,norm=Norm.BATCH))

        if not start_with_relu:
            rep = rep[1:]

        if stride != 1:
            rep.append(SeparableConv2d_same(planes, planes, 3, stride=2))

        if stride == 1 and is_last:
            rep.append(SeparableConv2d_same(planes, planes, 3, stride=1))

        self.rep = nn.Sequential(*rep)

    def forward(self, inp):
        x = self.rep(inp)

        if self.skip is not None:
            skip = self.skip(inp)
        else:
            skip = inp

        x += skip

        return x



In [ ]:

class Xception(nn.Module):
    """
    Modified Aligned Xception
    """

    def __init__(
        self,
        dim: int = 3,
        in_chns: int = 1,
        out_chns: int = 4,
    ):

        super(Xception, self).__init__()

        entry_block3_stride = 2
        middle_block_dilation = 1
        exit_block_dilations = (1, 2)

        # entry flow
        self.conv1 = Convolution(dim, in_chns, 32, kernel_size=3,bias=False, strides=2, padding=1,norm=Norm.BATCH)
        self.relu = Act[Act.RELU](inplace=True)

        self.conv2 = Convolution(dim, 32, 64, kernel_size=3,bias=False, strides=1, padding=1,norm=Norm.BATCH)

        self.block1 = Block(64, 128, reps=2, stride=2, start_with_relu=False)
        self.block2 = Block(128, 256, reps=2, stride=2, start_with_relu=True, grow_first=True)
        self.block3 = Block(256, 728, reps=2, stride=entry_block3_stride, start_with_relu=True, grow_first=True,
                            is_last=True)

        # Middle flow
        self.middle_flow = nn.Sequential(
                *[Block(728, 728, reps=3, stride=1, dilation=middle_block_dilation, start_with_relu=True, grow_first=True)
                for _ in range(16)]
            )

        # Exit flow
        self.block20 = Block(728, 1024, reps=2, stride=1, dilation=exit_block_dilations[0],
                             start_with_relu=True, grow_first=False, is_last=True)

        self.conv3 = SeparableConv2d_same(1024, 1536, 3, stride=1, dilation=exit_block_dilations[1],norm=Norm.BATCH)

        self.conv4 = SeparableConv2d_same(1536, 1536, 3, stride=1, dilation=exit_block_dilations[1],norm=Norm.BATCH)

        self.conv5 = SeparableConv2d_same(1536, 2048, 3, stride=1, dilation=exit_block_dilations[1],norm=Norm.BATCH)



    def forward(self, x):
        # Entry flow
        x = self.conv1(x)
        x = self.relu(x)

        x = self.conv2(x)
        x = self.relu(x)

        x = self.block1(x)
        low_level_feat = x
        x = self.block2(x)
        x = self.block3(x)

        # Middle flow
        x = self.middle_flow(x)

        # Exit flow
        x = self.block20(x)
        x = self.conv3(x)
        x = self.relu(x)

        x = self.conv4(x)
        x = self.relu(x)

        x = self.conv5(x)
        x = self.relu(x)

        return x, low_level_feat



In [ ]:

class ASPP_module(nn.Module):
    def __init__(self, inplanes, planes, dilation):
        super(ASPP_module, self).__init__()
        dim = 3
        if dilation == 1:
            kernel_size = 1
            padding = 0
        else:
            kernel_size = 3
            padding = dilation
        self.atrous_convolution = Convolution(dim, inplanes, planes, kernel_size=kernel_size,
                                              strides=1, padding=padding, dilation=dilation, bias=False,norm=Norm.BATCH)
        self.relu = Act[Act.RELU]()


    def forward(self, x):
        x = self.atrous_convolution(x)

        return self.relu(x)




In [ ]:

class Deeplab(nn.Module):
    def __init__(
        self,
        dimensions: int = 3,
        in_channels: int = 1,
        out_channels: int = 4,
    ):

        super(Deeplab, self).__init__()
        self.xception_features = Xception(dimensions, in_channels, out_channels)

        # ASPP
        dilations = [1, 6, 12, 18]
        self.aspp1 = ASPP_module(2048, 256, dilation=dilations[0])
        self.aspp2 = ASPP_module(2048, 256, dilation=dilations[1])
        self.aspp3 = ASPP_module(2048, 256, dilation=dilations[2])
        self.aspp4 = ASPP_module(2048, 256, dilation=dilations[3])

        self.relu = Act[Act.RELU]()
        pool_type: Type[Union[nn.MaxPool2d, nn.MaxPool3d]] = Pool[Pool.ADAPTIVEAVG, dimensions]
        self.global_avg_pool = nn.Sequential(pool_type(1),
                                             Convolution(dimensions, 2048, 256, kernel_size=1, strides=1, bias=False,norm=Norm.BATCH),
                                             Act[Act.RELU]())

        self.conv1 = Convolution(dimensions, 1280, 256, kernel_size=1, bias=False,norm=Norm.BATCH)

        # adopt [1x1, 48] for channel reduction.
        self.conv2 = Convolution(dimensions, 128, 48, kernel_size=1, bias=False,norm=Norm.BATCH)

        self.last_conv = nn.Sequential(Convolution(dimensions, 304, 256, kernel_size=3, strides=1, padding=1, bias=False,norm=Norm.BATCH),
                                       Act[Act.RELU](),
                                       Convolution(dimensions, 256, 256, kernel_size=3,
                                                   strides=1, padding=1, bias=False,norm=Norm.BATCH),
                                       Act[Act.RELU](),
                                       Convolution(dimensions, 256, out_channels, kernel_size=1, strides=1))

    def forward(self, input):

        x, low_level_features = self.xception_features(input)
        x1 = self.aspp1(x)
        x2 = self.aspp2(x)
        x3 = self.aspp3(x)
        x4 = self.aspp4(x)
        x5 = self.global_avg_pool(x)
        x5 = F.interpolate(x5, size=(x4.size()[2],x4.size()[3],x4.size()[4]), mode='trilinear', align_corners=True)
        x = torch.cat((x1, x2, x3, x4, x5), dim=1)

        x = self.conv1(x)
        x = self.relu(x)
        x = F.interpolate(x, size=(int(math.ceil(input.size()[-3] / 4)), int(math.ceil(input.size()[-2] / 4)),
                                   int(math.ceil(input.size()[-1] / 4))), mode='trilinear', align_corners=True)

        low_level_features = self.conv2(low_level_features)
        low_level_features = self.relu(low_level_features)

        x = torch.cat((x, low_level_features), dim=1)
        x = self.last_conv(x)
        x = F.interpolate(x, size=(input.size()[2],input.size()[3],input.size()[4]), mode='trilinear', align_corners=True)

        return x


In [14]:
#| hide
import nbdev; nbdev.nbdev_export()